In [1]:
import os
os.chdir("..")
from t2Interp.T2I import T2IModel
from t2Interp.concept_search import KSteer
from t2Interp.mapper import MLPMapper
from utils.utils import preprocess_image
import torch as th
from reporting.config_loader import load_config, wandb_init_kwargs
from utils.runningstats import WandbUpdater
from utils.training import TrainingSpec, Training
from PIL import Image

/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `PYTORCH_TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
model = T2IModel("CompVis/stable-diffusion-v1-4", device="cuda:0", dtype="float16")

2025-10-15 20:31:39.152 | INFO     | t2Interp.T2I:__init__:105 - Enforcing eager attention implementation for attention pattern tracing. The HF default would be to use sdpa if available. To use sdpa, set attn_implementation='sdpa' or None to use the HF default.
Keyword arguments {'attn_implementation': 'eager', 'tokenizer_kwargs': {}, 'trust_remote_code': False} are not expected by StableDiffusionPipeline and will be ignored.
Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00, 14.03it/s]


In [ ]:
preprocess_fn = lambda x: preprocess_image(x, 512)
race_labels={"East Asian":0,"Indian":1,"Black":2,"White":3,"Middle Eastern":4,"Latino_Hispanic":5,"Southeast Asian":6}
gt_processing_fn = lambda x: race_labels[x]

In [4]:
model._wrappers

{'text_encoder_2': blocks:
 0: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 1: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 2: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 3: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 4: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 5: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 6: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 7: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 8: in_ | attn_in | attn_out | WO_in 

### check latent shape

In [5]:
cache = model.run_with_cache([""], accessors= [model.unet_2.down_attn_blocks[0].self_attn_out], **{"num_inference_steps":1, "seed":40})
for k,v in cache.items():
    print(k, v.output.shape)

100%|██████████| 1/1 [00:00<00:00,  2.93it/s]
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


unet_down_block_attn0_self_attn_out torch.Size([2, 4096, 320])


In [6]:
kwargs = {
    "preprocess_fn": preprocess_fn,
    "gt_processing_fn" : gt_processing_fn,
    "val_split":"val",
    "dataset_column": "image",
    "ground_truth_column":"race",
    "subset": 100,
    "use_val": True,
    "steps":100,
    "denoising_step":0,
    "training_device" : "cuda:0",
    "data_device" : "cpu",
    "autocast_dtype" : th.bfloat16,
    "d_submodule": 4096*320,
    "log_steps": 1,
    "refresh_batch_size":64,  
    "out_batch_size":16,
}

dataset = "nirmalendu01/fairface-trainval"
accessor = model.unet_2.down_attn_blocks[0].self_attn_out
mapper = MLPMapper(input_dim=4096*320, hidden_dim=4096, output_dim=7, dtype=th.float16, device="cuda:0")
loss_fn = th.nn.CrossEntropyLoss()
optimizers = [th.optim.Adam(mapper.parameters(), lr=1e-4)]
steering = KSteer(model)

wandb_config = load_config("reporting/config.yaml")
wandb_config["wandb"].update({"run_name" : "training_race_steering_mlp"})
init_kwargs = wandb_init_kwargs(wandb_config)
stats_updaters = [WandbUpdater(init_kwargs=init_kwargs)]

spec = TrainingSpec(name="training_race_steering_mlp", fn = steering.fit, stats_updaters=stats_updaters, args=(dataset, accessor, mapper,loss_fn ,optimizers), kwargs=kwargs)
Training(spec).run_trainer()

wandb: Currently logged in as: nirmalendu12345 (nirmalendu12345-sutd-singapore-university-of-technology-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
<class 'PIL.JpegImagePlugin.Jpeg

TypeError: 'NoneType' object is not iterable